# List-aware · CatBoost × XGBoost model showdown

**Goal:** compare our historically strongest list-aware approaches on **exactly the same chronological properties**, including hybrids of CatBoost Huber with XGBoost's best spline/geographic feature representations and fixed-weight multi-model blends. No neural networks.

The previously recorded **4.06% MdAPE** CatBoost Huber score used a different, full-scale historical evaluation. **Do not compare that number directly to a capped pilot.** This notebook fits a matched CatBoost Huber control in the same pilot.

The experiment delegates training to `listaware_shotgun.py` and uses the parameter configurations in `models/listaware_shotgun_specs.py`. Completed individual predictions are cached per member and month; rerunning with unchanged settings resumes without refitting them.

Full guide: [listaware_shotgun.md](../docs/listaware_shotgun.md).

## 1 · Repository and execution environment

Use a Python kernel with the project's dependencies. Local Windows/VS Code works from the project folder or `notebooks/`. Google Colab can clone the public code; it needs the normal competition listings dataset (cached/downloaded by `data.py`).

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

PROJECT_ROOT = None  # Optional manual repository checkout path.
DATA_DIR_OVERRIDE = None  # None preserves current environment; important for Parquet parity.
CLONE_IN_COLAB_IF_MISSING = True
INSTALL_REQUIREMENTS_IN_COLAB = True

def locate_project():
    if PROJECT_ROOT is not None:
        path = Path(PROJECT_ROOT).expanduser().resolve()
        if not (path / "listaware_shotgun.py").is_file():
            raise FileNotFoundError(f"Not a project checkout: {path}")
        return path
    current = Path.cwd().resolve()
    for parent in (current, *current.parents):
        for candidate in (parent, parent / "SSU-Data-Dash-hdnl-comp"):
            if (candidate / "listaware_shotgun.py").is_file():
                return candidate
    if "google.colab" in sys.modules and CLONE_IN_COLAB_IF_MISSING:
        target = Path("/content/SSU-Data-Dash-hdnl-comp")
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/HiddenL123/SSU-Data-Dash-hdnl-comp.git",
             str(target)],
            check=True,
        )
        return target
    raise FileNotFoundError("Open from the repository or set PROJECT_ROOT.")

ROOT = locate_project()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if DATA_DIR_OVERRIDE:
    os.environ["DATA_DIR"] = str(Path(DATA_DIR_OVERRIDE).expanduser().resolve())
if "google.colab" in sys.modules and INSTALL_REQUIREMENTS_IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(ROOT / "requirements.txt")], check=True)
print("Repository:", ROOT)
print("Python:", sys.version.split()[0])
print("DATA_DIR:", os.environ.get("DATA_DIR", "(project data/ or Hugging Face)"))

## 2 · Inspect the recorded candidates and fixed blends

The key **cross-family transfer** is CatBoost Huber trained on the feature set developed for the strong XGBoost off-market model—not simply averaging the original models' predictions. Other arms train tuned XGBoost or LightGBM on their own backends.

Additional blended arms average **predicted sale prices in log space** using weights fixed before seeing these fold results.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
from models.listaware_shotgun_specs import CANDIDATES, DEFAULT_CANDIDATES, BLENDS

catalog = pd.DataFrame([{
    "candidate": key, "backend": c.backend, "target": c.target,
    "loss": c.loss, "feature_set": c.feature_name,
    "tuned": c.tuned, "in_default": key in DEFAULT_CANDIDATES,
} for key, c in CANDIDATES.items()])
display(catalog)
blend_catalog = pd.DataFrame([
    {"blend": name, "members": ", ".join(f"{m}: {w:.0%}" for m, w in weights.items())}
    for name, weights in BLENDS.items()
])
display(blend_catalog)

## 3 · Select the experiment

Set `SCREEN = "broad"` for the default multi-family shotgun; set `"cross_family"` for a smaller CatBoost/XGBoost-specific test, or `"objectives"` to isolate the target and loss.

The default pilot uses **August–October 2025**, 10,000 training properties per month, 350 validation homes per month, and at most 300 boosting rounds per model. It uses exactly matched evaluation rows and the same chronological training restrictions for every candidate.

If you change models, months or budgets after an earlier run, change `OUTPUT_DIR` as well. Model caches are keyed by a strict experiment manifest.

In [ ]:
from pathlib import Path
from listaware_shotgun import run_listaware_shotgun

SCREEN = "broad"  # "broad", "cross_family", "objectives"
SCREENS = {
    "broad": tuple(DEFAULT_CANDIDATES),
    "cross_family": (
        "cat_huber_baseline", "cat_huber_spline_geo", "xgb_spline_geo",
        "lgb_spline_geo",
    ),
    "objectives": (
        "cat_huber_baseline", "cat_huber_direct", "cat_mae",
        "cat_rmse", "cat_tuned_rmse",
    ),
}
CANDIDATES_TO_TEST = SCREENS[SCREEN]
MONTHS = ("2025-08", "2025-09", "2025-10")
MODE = "expanding"  # Optionally "fixed_origin" with a DIFFERENT output folder.
OUTPUT_DIR = Path(f"outputs/listaware_notebook_{SCREEN}_{MODE}_pilot")

RUN_TESTS = False
RUN_PILOT = True
WITH_BLENDS = True
TRAIN_CAP = 10_000
VALIDATION_CAP = 350
TREE_CAP = 300
THREADS = 4
BOOTSTRAP_REPEATS = 300
SEED = 42

assert "cat_huber_baseline" in CANDIDATES_TO_TEST
assert set(CANDIDATES_TO_TEST).issubset(CANDIDATES)
print("Candidate count:", len(CANDIDATES_TO_TEST))
print("Active fixed blends:", [name for name, w in BLENDS.items()
                              if WITH_BLENDS and set(w).issubset(CANDIDATES_TO_TEST)])
print("Output:", OUTPUT_DIR)
display(catalog.loc[catalog.candidate.isin(CANDIDATES_TO_TEST)])

## 4 · Input and regression-test preflight

The competition's historic `listings` file supplies all labels. Validation homes are passed to the model **without `sold_price` and `date_sold`**, but their **list prices remain visible**. Training includes only sales completed before the prediction month's cutoff.

The tests are optional in the notebook because the full experiment already validates schema and cohort alignment; run them at least once locally before relying on results.

In [ ]:
import data

if RUN_TESTS:
    subprocess.run([
        sys.executable, "-m", "pytest",
        "tests/test_listaware_shotgun.py",
        "tests/test_realworld_avm_shotgun.py", "-q",
    ], check=True, cwd=ROOT)

print("Listings source:", data.find_file("listings"))
print("Train cutoff: earlier completed sales only")
print("Screen:", SCREEN)
print("Evaluation months:", MONTHS)
print("Target reconstruction: exp(predicted log sale/list ratio) * list_price")

## 5 · Run or resume the pilot

The first execution fits each candidate and saves a separate CSV **immediately after each completed member/month**. Rerun with identical settings to resume. `RUN_PILOT = False` loads completed CSVs without retraining.

In [ ]:
if RUN_PILOT:
    pooled, monthly, by_type, predictions = run_listaware_shotgun(
        months=MONTHS,
        candidates=CANDIDATES_TO_TEST,
        full=False,
        mode=MODE,
        train_cap=TRAIN_CAP,
        validation_cap=VALIDATION_CAP,
        trees=TREE_CAP,
        threads=THREADS,
        seed=SEED,
        with_blends=WITH_BLENDS,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
        output_dir=OUTPUT_DIR,
    )
else:
    pooled = pd.read_csv(OUTPUT_DIR / "pooled_scores.csv")
    monthly = pd.read_csv(OUTPUT_DIR / "monthly_scores.csv")
    by_type = pd.read_csv(OUTPUT_DIR / "house_type_scores.csv")
    predictions = pd.read_csv(OUTPUT_DIR / "predictions.csv",
                              dtype={"ml_num": str})
print("Prediction rows:", len(predictions))

## 6 · Leaderboard: models and hybrids

`delta_mdape_pp` is relative to **CatBoost Huber trained and scored on the identical pilot cohort**. Negative means better. MdAPE is the primary competition metric, with MAPE and tail behaviour useful for secondary evaluation.

Use `delta_mdape_ci_low` / `delta_mdape_ci_high` as an exploratory paired bootstrap diagnostic, not a guarantee of an untouched test win.

In [ ]:
columns = [
    "arm", "variant", "homes", "mdape", "mape", "p90_ape",
    "delta_mdape_pp", "delta_mape_pp",
    "homes_improved_fraction",
    "delta_mdape_ci_low", "delta_mdape_ci_high",
]
display(pooled[[c for c in columns if c in pooled]].round(4).reset_index(drop=True))
reference = pooled.loc[
    (pooled.arm == "model") & (pooled.variant == "cat_huber_baseline")
]
assert len(reference) == 1
print("Matched CatBoost Huber baseline MdAPE:",
      f"{reference.mdape.iloc[0]:.4f}%")
best_row = pooled.sort_values(["mdape", "mape"]).iloc[0]
print(f"Best candidate: {best_row.variant} ({best_row.arm}) — "
      f"MdAPE {best_row.mdape:.4f}%")

In [ ]:
import matplotlib.pyplot as plt

ordered = pooled.sort_values("delta_mdape_pp")
fig, ax = plt.subplots(figsize=(9, max(4, .40 * len(ordered))))
labels = ordered["variant"].astype(str) + " [" + ordered["arm"].astype(str) + "]"
ax.barh(labels, ordered.delta_mdape_pp)
ax.axvline(0, color="gray", linewidth=1)
ax.invert_yaxis()
ax.set_title("List-aware: paired MdAPE difference vs CatBoost Huber")
ax.set_xlabel("Change in MdAPE (percentage points; negative = better)")
plt.tight_layout()
plt.show()

## 7 · Monthly and property-type stability

The pooled metric can mask a big win in just one month or one class of homes. For shortlist selection, prefer models whose paired improvement repeats across months and doesn't create a severe tail-risk trade-off.

In [ ]:
display(monthly.pivot_table(
    index=["arm", "variant"], columns="month", values="delta_mdape_pp"
).round(3))
if len(by_type):
    print("By house type (positive delta = worse than CatBoost Huber):")
    display(by_type.pivot_table(
        index=["arm", "variant"], columns="house_type_name",
        values="delta_mdape_pp",
    ).round(3))

## 8 · Do CatBoost and XGBoost actually complement one another?

Compare blended rows with the best individual members on the same homes. A blended model may have **lower MAPE/P90 but slightly higher MdAPE**; this matters because the competition emphasizes the median.

The recorded blends have *precommitted weights*. Do not optimize new weights on these same months and then describe that score as untouched validation.

In [ ]:
model_scores = pooled.loc[pooled.arm == "model"].sort_values(["mdape", "mape"])
blend_scores = pooled.loc[pooled.arm == "blend"].sort_values(["mdape", "mape"])
print("Best single models:")
display(model_scores[["variant", "mdape", "mape", "p90_ape", "delta_mdape_pp"]].head(10).round(4))
if len(blend_scores):
    print("Predeclared cross-family blends:")
    display(blend_scores[["variant", "mdape", "mape", "p90_ape", "delta_mdape_pp"]].round(4))
else:
    print("No complete predeclared blend was available with these candidates.")

# On the identical cohort, which homes does the top challenger improve?
control = predictions.loc[
    (predictions.arm == "model") & (predictions.variant == "cat_huber_baseline")
].set_index(["month", "ml_num"]).sort_index()
winner = predictions.loc[
    (predictions.arm == best_row.arm)
    & (predictions.variant == best_row.variant)
].set_index(["month", "ml_num"]).sort_index()
paired_error = pd.DataFrame({
    "baseline_ape": control["ape"],
    "candidate_ape": winner.loc[control.index, "ape"],
})
paired_error["improvement_pp"] = paired_error.baseline_ape - paired_error.candidate_ape
display(paired_error.describe(percentiles=[.1, .5, .9]).round(3))

## 9 · Full-strength shortlist (explicitly opt in)

After the pilot, choose two to four promising candidates and keep CatBoost Huber as the reference. Full mode uses all eligible historic sales and each model's recorded full boosting-round count (CatBoost 900, tuned XGBoost 700, tuned LightGBM 1,800). This requires more compute.

Run `--mode fixed_origin` with a **separate output folder** to approximate a model trained at one historical cutoff and applied to multiple months without refreshing.

The original 4.06% CatBoost result can only be fairly compared after evaluating the same held-out homes with the same chronology and fitting budget. Even then, December–May has been inspected during earlier development.

In [ ]:
RUN_FULL = False
FULL_CANDIDATES = (
    "cat_huber_baseline", "cat_huber_spline_geo",
    "xgb_spline_geo", "lgb_spline_geo",
)
FULL_MODE = "expanding"
FULL_OUTPUT_DIR = Path(f"outputs/listaware_notebook_full_{FULL_MODE}")

if RUN_FULL:
    full_pooled, full_monthly, full_by_type, full_predictions = run_listaware_shotgun(
        months=MONTHS, candidates=FULL_CANDIDATES,
        full=True, mode=FULL_MODE,
        threads=THREADS, with_blends=WITH_BLENDS,
        bootstrap_repeats=BOOTSTRAP_REPEATS, seed=SEED,
        output_dir=FULL_OUTPUT_DIR,
    )
    display(full_pooled.round(4))
else:
    print("Full-strength training disabled; set RUN_FULL=True to opt in.")

## 10 · Optional fixed-origin robustness check

The expanding study refits before each development month. To check a harder forecast horizon, you can freeze the training cutoff before August 2025 and predict August–October without updating model training between months.

This creates a **separate experiment**. Since the existing evaluation months have been repeatedly inspected, do not treat either study as an untouched holdout.

In [ ]:
RUN_FIXED_ORIGIN = False
FIXED_ORIGIN_OUTPUT = Path("outputs/listaware_notebook_fixed_origin_pilot")

if RUN_FIXED_ORIGIN:
    fixed_pooled, fixed_monthly, fixed_by_type, fixed_predictions = (
        run_listaware_shotgun(
            months=MONTHS,
            candidates=CANDIDATES_TO_TEST,
            full=False,
            mode="fixed_origin",
            train_cap=TRAIN_CAP, validation_cap=VALIDATION_CAP,
            trees=TREE_CAP, threads=THREADS,
            seed=SEED, with_blends=WITH_BLENDS,
            bootstrap_repeats=BOOTSTRAP_REPEATS,
            output_dir=FIXED_ORIGIN_OUTPUT,
        )
    )
    display(fixed_pooled.round(4))
else:
    print("Fixed-origin cross-check disabled.")

## 11 · Promotion checklist

Choose a shortlist by **paired MdAPE**, with MAPE/P90 and per-month consistency as tie-breakers. A successful CatBoost–XGBoost blend can become a candidate for a stronger, fully frozen list-aware ensemble.

**Not included yet:** feeding the full-history off-market model's *in-sample* predictions into list-aware training. That would leak sold prices. A separate two-stage experiment would require **out-of-time off-market predictions for every stage-two training observation** before creating the list-price-versus-market-value feature.

After the two notebook screens finish, send their `pooled_scores.csv` and `pooled_vs_ensemble.csv` results for a controlled winner analysis. The scripts do not overwrite your existing saved models or competition submissions.